# 9.10 · 正则化 / Regularization

> **课程定位 / Where this fits**
> 第 10 课，**Part 9 · 深度学习基础**。
> Lesson 10, **Part 9 · Deep Learning Foundations**.
>
> 深层网络参数极多，**很容易把训练集背下来(过拟合)**——训练损失很低，但在新数据上表现糟糕。**正则化(regularization)** 是一切抑制过拟合、提升泛化的手段：**L2 weight decay、Dropout、BatchNorm、数据增强、早停**。它们是深度学习里最实用、最常考的一类技巧。
> Deep nets have huge parameter counts and **easily memorize the training set (overfit)** — low train loss but poor on new data. **Regularization** is anything that curbs overfitting and improves generalization: **L2 weight decay, Dropout, BatchNorm, data augmentation, early stopping**. Among the most practical and frequently-asked DL topics.
>
> 💼 **实战/面试视角**："Dropout 原理 / 训练和推理为什么不同 / BatchNorm 为什么有效 / L2=weight decay" 是高频题。
> 💼 **Practical/interview angle:** "how Dropout works / why train vs inference differ / why BatchNorm helps / L2 = weight decay" — high frequency.

> 📐 **符号约定 / Notation**
> - $\lambda$ —— 正则化强度 / regularization strength
> - $p$ —— Dropout 丢弃概率 / Dropout drop probability

> 💡 **面试相关 / Interview-relevant**
> - "Dropout 原理 + 训练/推理差异"（出镜率 ★★★★★）
> - "L2 正则 = weight decay"（★★★★★）
> - "BatchNorm 为什么有效 + train/eval 差异"（★★★★★）
> - "L1 vs L2 区别"（★★★★）
> - "还有哪些正则化手段"（★★★★，早停/数据增强/标签平滑）

---

## 学习目标 / Learning Objectives
1. 直观看到过拟合，并理解正则化为何能缓解。
   See overfitting visually and why regularization helps.
2. 掌握 **L2 weight decay** 与 **L1** 的区别。
   Master L2 weight decay vs L1.
3. 理解 **Dropout** 的原理与训练/推理差异。
   Understand Dropout and its train/inference difference.
4. 理解 **BatchNorm** 为什么稳定并加速训练。
   Understand why BatchNorm stabilizes and speeds training.
5. 汇总其他手段：早停、数据增强、标签平滑。
   Summarize early stopping, data augmentation, label smoothing.

## 目录 / TOC
1. [先看过拟合 ⭐](#1)
2. [L2 weight decay 与 L1 ⭐](#2)
3. [Dropout ⭐](#3)
4. [BatchNorm ⭐](#4)
5. [其他手段 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 先看过拟合 ⭐ / Seeing Overfitting

**过拟合**：模型不仅学到了数据的真实规律，还把训练集里的**噪声**也死记硬背下来。表现是**训练损失持续下降，但验证损失先降后升**——两条曲线越分越开。
**Overfitting:** the model learns not only the true signal but also memorizes the **noise** in the training set. Symptom: **train loss keeps dropping while val loss drops then rises** — the two curves diverge.

我们故意用一个**过大的网络 + 很少的数据**来制造过拟合，之后再看正则化如何拉近这两条曲线。
We deliberately use an **oversized net + little data** to induce overfitting, then see regularization narrow the gap.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
sns.set_theme(style="whitegrid")

digits = load_digits()
# 故意只用很少训练样本 → 容易过拟合 / use very few training samples → easy to overfit
X_tr, X_te, y_tr, y_te = train_test_split(digits.data/16.0, digits.target, train_size=150,
                                          test_size=600, stratify=digits.target, random_state=0)
Xtr = torch.tensor(X_tr, dtype=torch.float32); ytr = torch.tensor(y_tr)
Xte = torch.tensor(X_te, dtype=torch.float32); yte = torch.tensor(y_te)

def run(build_net, weight_decay=0.0, epochs=300):
    torch.manual_seed(0); net = build_net()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=weight_decay)
    ce = nn.CrossEntropyLoss(); tr_hist, te_hist = [], []
    for _ in range(epochs):
        net.train(); opt.zero_grad(); ce(net(Xtr), ytr).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            tr_hist.append(ce(net(Xtr), ytr).item()); te_hist.append(ce(net(Xte), yte).item())
    return tr_hist, te_hist

big = lambda: nn.Sequential(nn.Linear(64,256), nn.ReLU(), nn.Linear(256,256), nn.ReLU(), nn.Linear(256,10))
tr, te = run(big)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(tr, label="训练损失"); ax.plot(te, label="验证损失")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(); ax.set_title("过拟合: 训练损失→0, 但验证损失先降后升(两线分开)")
plt.tight_layout(); plt.show()
print(f"训练损失 = {tr[-1]:.3f} (几乎背下来), 验证损失 = {te[-1]:.3f} (反而变差)")
print("两条线越分越开 = 典型过拟合; 下面用正则化拉近它们")


<a id="2"></a>
## 2. L2 weight decay 与 L1 ⭐ / L2 Weight Decay & L1

**L2 正则化**：在损失里加一项 $\lambda \sum w^2$，惩罚**大权重**。直觉是"奥卡姆剃刀"——**更小、更平滑的权重通常泛化更好**，因为模型不会对个别输入过度敏感。
**L2 regularization:** add $\lambda \sum w^2$ to the loss, penalizing **large weights**. Intuition is Occam's razor — **smaller, smoother weights usually generalize better**, the model isn't overly sensitive to individual inputs.

**关键事实(面试爱考)**：对 SGD 来说，**L2 正则 ≡ weight decay(权重衰减)**——每步更新前先把权重乘以一个略小于 1 的数。这就是为什么 PyTorch 优化器里那个参数直接叫 `weight_decay`。（注：对 Adam 二者不完全等价，所以才有 AdamW，见 9.7。）
**Key fact (loved in interviews):** for SGD, **L2 ≡ weight decay** — shrink weights by a factor slightly below 1 each step. That's why the PyTorch optimizer argument is literally `weight_decay`. (Note: for Adam they're not exactly equivalent — hence AdamW, see 9.7.)

**L1 正则化**：加 $\lambda \sum |w|$，倾向把一些权重压成**正好 0**，产生**稀疏**模型（自动特征选择）。L2 只是让权重变小但很少正好为 0。
**L1 regularization:** add $\lambda \sum |w|$, pushing some weights to **exactly 0**, yielding a **sparse** model (automatic feature selection). L2 shrinks but rarely zeroes.


In [ ]:
# 给过拟合网络加上 L2 weight decay, 看验证损失是否改善 / add L2 weight decay
tr0, te0 = run(big, weight_decay=0.0)
tr2, te2 = run(big, weight_decay=1e-2)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(te0, label="无正则 (验证)", lw=2)
ax.plot(te2, label="L2 weight_decay=1e-2 (验证)", lw=2)
ax.set_xlabel("epoch"); ax.set_ylabel("验证损失"); ax.legend()
ax.set_title("L2 weight decay: 惩罚大权重 → 缓解过拟合, 验证损失更低更稳")
plt.tight_layout(); plt.show()
print(f"无正则:  最终验证损失 = {te0[-1]:.3f}")
print(f"加 L2:   最终验证损失 = {te2[-1]:.3f}  ← 通常更低/更平稳")
print("记忆: 对 SGD, L2 正则 ≡ weight decay; L1 产生稀疏(权重正好=0), L2 只是变小")


<a id="3"></a>
## 3. Dropout ⭐ / Dropout

**Dropout** 是深度学习特有的正则化：**训练时，每个神经元以概率 $p$ 被随机"丢弃"（输出置 0）**。每个 mini-batch 丢弃的神经元都不同，相当于每次都在训练一个**不同的、更小的子网络**。
**Dropout** is a DL-specific regularizer: **during training, each neuron is randomly "dropped" (set to 0) with probability $p$**. Different neurons drop each mini-batch, so you train a **different, smaller sub-network** each time.

**为什么有效**：神经元不能依赖某个特定的"队友"（它随时可能消失），被迫学习**更鲁棒、更冗余**的特征。也可理解为**对指数多个子网络做集成**(类似 bagging)。
**Why it works:** a neuron can't rely on a specific "teammate" (it may vanish anytime), forcing **more robust, redundant** features. Also viewable as **ensembling exponentially many sub-networks** (bagging-like).

**训练 vs 推理（高频考点）**：训练时随机丢弃；**推理(eval)时不丢弃**，用全部神经元。为了让推理时的期望输出与训练一致，PyTorch 用 **inverted dropout**：训练时把保留的神经元除以 $(1-p)$ 放大。所以**一定要用 `model.train()` / `model.eval()` 切换模式**，否则推理会出错。
**Train vs inference (high-frequency):** drop randomly in training; **no dropping at inference (eval)** — use all neurons. To match expected outputs, PyTorch uses **inverted dropout**: scale kept neurons by $1/(1-p)$ in training. So **always toggle `model.train()` / `model.eval()`**, or inference breaks.


In [ ]:
# 手动演示 inverted dropout 的训练/推理差异 / demo inverted dropout train vs eval
torch.manual_seed(0)
drop = nn.Dropout(p=0.5)
x = torch.ones(1, 10)                                       # 全 1 输入便于观察 / all-ones input
drop.train(); out_train = drop(x)                          # 训练模式: 随机置0并把保留的×2 / random zero + scale by 1/(1-p)
drop.eval();  out_eval = drop(x)                           # 推理模式: 原样通过 / pass through unchanged
print("训练模式输出 (随机丢一半, 保留的被放大到 2):"); print(out_train.numpy())
print("推理模式输出 (不丢弃, 原样):"); print(out_eval.numpy())
print(f"训练输出均值 = {out_train.mean():.2f}, 推理输出均值 = {out_eval.mean():.2f}  ← 期望一致(inverted dropout)")

# 在过拟合网络中加 Dropout / add Dropout to the overfitting net
def big_dropout():
    return nn.Sequential(nn.Linear(64,256), nn.ReLU(), nn.Dropout(0.5),
                         nn.Linear(256,256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256,10))
trd, ted = run(big_dropout)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(te0, label="无正则 (验证)", lw=2); ax.plot(ted, label="Dropout 0.5 (验证)", lw=2)
ax.set_xlabel("epoch"); ax.set_ylabel("验证损失"); ax.legend(); ax.set_title("Dropout: 随机丢神经元→学更鲁棒特征→缓解过拟合")
plt.tight_layout(); plt.show()
print(f"\n无正则验证损失={te0[-1]:.3f}, Dropout验证损失={ted[-1]:.3f}")
print("关键: 必须 model.train()/model.eval() 切换; 训练丢弃, 推理用全部神经元")


<a id="4"></a>
## 4. BatchNorm ⭐ / Batch Normalization

**Batch Normalization(批归一化)**：在每一层之后，**用当前 mini-batch 的均值和方差把激活值标准化**（减均值除标准差），再用两个可学习参数 $\gamma, \beta$ 缩放平移。
**Batch Normalization:** after a layer, **standardize activations using the current mini-batch's mean and variance** (subtract mean, divide by std), then scale/shift with learnable $\gamma, \beta$.

**为什么有效**（面试要点）：(1) 让每层输入分布更稳定，**减轻内部协变量偏移**，训练更快更稳；(2) 允许用**更大的学习率**；(3) 有**轻微正则化**效果（每个样本的归一化依赖于 batch 里的其他样本，引入噪声）。它常常让深网络从"训不动"变成"轻松训练"。
**Why it works** (interview): (1) keeps each layer's input distribution stable, **reducing internal covariate shift**, faster/stabler training; (2) allows **larger learning rates**; (3) provides **mild regularization** (each sample's normalization depends on the batch, adding noise). Often turns "untrainable" deep nets into "easy to train."

**train/eval 差异（必考）**：训练时用**当前 batch** 的统计量，同时维护一个**移动平均**；推理时用这个**移动平均**（因为推理可能只有 1 个样本，没法算 batch 统计量）。所以 BatchNorm 也必须 `train()/eval()` 切换。
**train/eval difference (must-know):** training uses the **current batch** statistics while maintaining a **running average**; inference uses that **running average** (inference may have just 1 sample, no batch stats). So BatchNorm also needs `train()/eval()` toggling.


In [ ]:
# BatchNorm 加速并稳定较深网络的训练 / BatchNorm speeds & stabilizes a deeper net
def deep_plain():
    return nn.Sequential(nn.Linear(64,128), nn.ReLU(), nn.Linear(128,128), nn.ReLU(),
                         nn.Linear(128,128), nn.ReLU(), nn.Linear(128,10))
def deep_bn():
    return nn.Sequential(nn.Linear(64,128), nn.BatchNorm1d(128), nn.ReLU(),
                         nn.Linear(128,128), nn.BatchNorm1d(128), nn.ReLU(),
                         nn.Linear(128,128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128,10))
# 用全量训练集对比收敛速度 / use full training set to compare convergence speed
Xtr2 = torch.tensor(digits.data/16.0, dtype=torch.float32); ytr2 = torch.tensor(digits.target)
def train_loss_curve(build, epochs=80):
    torch.manual_seed(0); net = build(); opt = torch.optim.SGD(net.parameters(), lr=0.1); ce = nn.CrossEntropyLoss(); h=[]
    for _ in range(epochs):
        net.train(); opt.zero_grad(); loss = ce(net(Xtr2), ytr2); loss.backward(); opt.step(); h.append(loss.item())
    return h
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(train_loss_curve(deep_plain), label="无 BatchNorm", lw=2)
ax.plot(train_loss_curve(deep_bn), label="有 BatchNorm", lw=2)
ax.set_xlabel("epoch"); ax.set_ylabel("训练损失"); ax.legend(); ax.set_title("BatchNorm: 稳定每层输入分布 → 收敛更快(可用更大 LR)")
plt.tight_layout(); plt.show()
print("BatchNorm: 标准化每层激活→减轻内部协变量偏移→更快更稳, 可用更大 LR, 还有轻微正则")
print("train: 用当前 batch 统计量+维护移动平均; eval: 用移动平均 → 必须 train()/eval() 切换")


<a id="5"></a>
## 5. 其他手段 + 小结 ⭐ / Other Methods & Summary

除了上面四种，还有几类常用正则化（面试可补充）：
Besides the four above, several other common regularizers (good to mention in interviews):
- **早停(Early stopping)**：监控验证损失，一旦开始上升就停（见 7.3）。最简单有效，几乎零成本。
  **Early stopping:** watch val loss; stop once it rises (see 7.3). Simplest and nearly free.
- **数据增强(Data augmentation)**：对训练样本做随机变换（图像翻转/裁剪/旋转、文本同义替换），等于免费扩充数据，是 CV/NLP 的标配（Part 10 详讲）。
  **Data augmentation:** random transforms (image flip/crop/rotate, text synonyms) — free data expansion, standard in CV/NLP (detailed in Part 10).
- **标签平滑(Label smoothing)**：把硬标签 (0/1) 换成软标签 (如 0.1/0.9)，防止模型对预测过度自信。
  **Label smoothing:** replace hard labels (0/1) with soft ones (e.g. 0.1/0.9), preventing over-confidence.
- **权重正则的本质**都是给模型加约束、降低有效容量，迫使它学真正的规律而非噪声。
  All of these add constraints / reduce effective capacity, forcing the model to learn signal not noise.

下面用早停做个收尾演示。
A quick early-stopping demo to close.


In [ ]:
# 早停: 记录验证损失最低的轮次 / early stopping: find epoch of lowest val loss
tr, te = run(big, epochs=300)
best_epoch = int(np.argmin(te))                            # 验证损失最低的 epoch / epoch with min val loss
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(te, label="验证损失", lw=2)
ax.axvline(best_epoch, color="C3", ls="--", label=f"早停点 epoch={best_epoch}")
ax.set_xlabel("epoch"); ax.set_ylabel("验证损失"); ax.legend(); ax.set_title("早停: 在验证损失最低处停下, 之后再训只会过拟合")
plt.tight_layout(); plt.show()
print(f"验证损失在 epoch={best_epoch} 处最低 ({te[best_epoch]:.3f}); 之后继续训只会过拟合")
print("早停 = 最简单的正则化: 监控验证集, 一升就停, 保存最佳权重")


```
过拟合: 训练损失↓但验证损失先↓后↑(背下噪声); 正则化=抑制过拟合提升泛化
L2 (weight decay): 惩罚 Σw² → 权重更小更平滑; 对SGD L2≡weight decay; Adam 用 AdamW
L1: 惩罚 Σ|w| → 稀疏(权重正好0, 自动特征选择)
Dropout: 训练随机丢神经元(学鲁棒特征/集成); 推理不丢; inverted dropout; 必须 train()/eval()
BatchNorm: 用batch均值方差标准化激活→减轻协变量偏移→快/稳/可大LR/轻微正则; train用batch+移动平均, eval用移动平均
其他: 早停(最简单) / 数据增强 / 标签平滑
```

### 💡 面试速查 / Interview cheat-sheet
1. **L2=weight decay** (对SGD), 惩罚大权重; **L1→稀疏**。
   L2 = weight decay (SGD), penalizes large weights; L1 → sparsity.
2. **Dropout**: 训练随机丢/推理不丢, inverted dropout, 必须切 train()/eval()。
   Dropout: drop in train / keep in eval, inverted dropout, must toggle train()/eval().
3. **BatchNorm**: 标准化激活, 减轻协变量偏移, 更快更稳; train/eval 统计量不同。
   BatchNorm: normalize activations, reduce covariate shift, faster; different stats train/eval.
4. **早停**: 验证损失一升就停, 最简单有效。
   Early stopping: stop when val loss rises, simplest.
5. **数据增强/标签平滑**: 其他常用手段。
   Data augmentation / label smoothing: other common methods.

### 下一节 / Next
**9.11 训练技巧**——把前几节串成一套实战训练流程: 梯度裁剪、梯度累积、混合精度、checkpoint、监控等让训练又快又稳的工程技巧。
**9.11 Training Tricks** — chaining everything into a practical training workflow: gradient clipping, gradient accumulation, mixed precision, checkpointing, monitoring.
